## 01 - Download TCGA-PAAD RNA-seq (GDC, STAR - Counts)

### Aim
* Query the GDC API for TCGA-PAAD harmonized RNA-seq gene expression quantification (STAR - Counts workflow) and download the per-sample files, then assemble them into a single `SummarizedExperiment`.

### Input
* GDC API: project `TCGA-PAAD`, `data.category = "Transcriptome Profiling"`, `data.type = "Gene Expression Quantification"`, `workflow.type = "STAR - Counts"`, `sample.type = c("Primary Tumor", "Solid Tissue Normal")`.
* Note: this deliberately does **not** reuse `progress/gdc_manifest...txt`, which was built from "add project to cart" on the GDC portal and contains ~13,000 files across every TCGA-PAAD data type (WGS BAMs, VCFs, SV calls, IDATs, RPPA, etc. - on the order of 100+ TB). We only need the RNA-seq gene expression quantification files, so we re-query the GDC API directly for just that data type.

### Output
* Per-sample GDC gene-quantification files under `data/raw/TCGA-PAAD/Transcriptome_Profiling/Gene_Expression_Quantification/`.
* `data/raw/TCGA-PAAD_RNAseq_SE.rds`: a `SummarizedExperiment` (genes x samples) with assays `tpm_unstrand`, `unstranded` (raw counts), `fpkm_unstrand`, `fpkm_uq_unstrand`, plus row (gene) and column (sample) metadata. The STAR - Counts gene-level file already contains a GDC-computed TPM column (`tpm_unstrand`) - this is the raw (non-batch-corrected, non-log) TPM matrix used downstream.

### Steps
* Build the `GDCquery` for TCGA-PAAD RNA-seq gene expression quantification.
* Inspect the query result (file count, case count, sample type breakdown) before downloading anything.
* `GDCdownload` the files (resumable - re-running skips files already present).
* `GDCprepare` into a `SummarizedExperiment` and save it to disk.

### Result
* 182 files found/downloaded: 178 Primary Tumor + 4 Solid Tissue Normal (182 unique cases), ~771 MB total.
* `SummarizedExperiment`: 60,660 genes x 182 samples, assays `unstranded, stranded_first, stranded_second, tpm_unstrand, fpkm_unstrand, fpkm_uq_unstrand`.

### Conclusion
* Raw harmonized RNA-seq data (including GDC's own TPM calculation) is retrieved and cached locally in `data/raw/`, ready for matrix assembly in notebook 02.


### Set working directory to the project root

In [1]:
# Jupyter's R kernel starts with the notebook's own folder as the working
# directory. This pipeline's root is progress/ (scripts/ and data/ live
# nested under progress/, not the repo root) - every path below is
# relative to that.
if (basename(getwd()) == "rnaseq_download") setwd(file.path("..", ".."))
getwd()

[1] "/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer"

### Load packages and set the raw-data directory

In [2]:
library(TCGAbiolinks)
library(SummarizedExperiment)

raw_dir <- "data/raw"
dir.create(raw_dir, showWarnings = FALSE, recursive = TRUE)

Loading required package: MatrixGenerics



Loading required package: matrixStats




Attaching package: ‘MatrixGenerics’




The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges, rowRanks,
    rowSdDiffs, rowSds, rowSums2, rowTabulates, rowVarDiffs, rowVars,
    rowWeightedMads, rowWeightedMeans, rowWeightedMedians,
    rowWeightedSds, rowWeig

Loading required package: GenomicRanges



Loading required package: stats4



... (14 outputs left out)

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.





Attaching package: ‘Biobase’




The following object is masked from ‘package:MatrixGenerics’:

    rowMedians




The following objects are masked from ‘package:matrixStats’:

    anyMissing, rowMedians




### Build the GDC query for TCGA-PAAD RNA-seq gene expression quantification

In [3]:
query <- GDCquery(
  project       = "TCGA-PAAD",
  data.category = "Transcriptome Profiling",
  data.type     = "Gene Expression Quantification",
  workflow.type = "STAR - Counts",
  sample.type   = c("Primary Tumor", "Solid Tissue Normal")
)

--------------------------------------



o GDCquery: Searching in GDC database



--------------------------------------



Genome of reference: hg38



--------------------------------------------



oo Accessing GDC. This might take a while...



... (12 outputs left out)

ooo Checking if there are results for the query



-------------------



o Preparing output



-------------------



### Inspect what will be downloaded before pulling any files

In [4]:
plan <- getResults(query)
cat(sprintf(
  "Found %d files | %d unique cases | sample types: %s\n",
  nrow(plan),
  length(unique(plan$cases)),
  paste(table(plan$sample_type), names(table(plan$sample_type)), collapse = "; ")
))

Found 182 files | 182 unique cases | sample types: 178 Primary Tumor; 4 Solid Tissue Normal


### Download (resumable: re-running skips files already present in `raw_dir`)

In [5]:
GDCdownload(
  query          = query,
  directory      = raw_dir,
  method         = "api",
  files.per.chunk = 20
)

Of the 182 files for download 182 already exist.



All samples have been already downloaded



### Assemble into a `SummarizedExperiment`
Genes x samples, with TPM / counts / FPKM assays plus row (gene) and column (sample) metadata attached.

In [6]:
rna_se <- GDCprepare(
  query     = query,
  directory = raw_dir,
  save      = TRUE,
  save.filename = file.path(raw_dir, "TCGA-PAAD_RNAseq_SE.rds")
)

cat(sprintf(
  "SummarizedExperiment: %d genes x %d samples. Assays: %s\n",
  nrow(rna_se), ncol(rna_se), paste(assayNames(rna_se), collapse = ", ")
))

saveRDS(rna_se, file.path(raw_dir, "TCGA-PAAD_RNAseq_SE.rds"))

|                                                    |  0%                      

|                                                    |0.5494505% ~7 s remaining 

|                                                    |1.098901% ~6 s remaining  

|                                                    |1.648352% ~6 s remaining  

|=                                                   |2.197802% ~5 s remaining  

|=                                                   |2.747253% ~5 s remaining  

... (184 outputs left out)

Available assays in SummarizedExperiment : 
  => unstranded
  => stranded_first
  => stranded_second
  => tpm_unstrand
  => fpkm_unstrand
  => fpkm_uq_unstrand



=> Saving file: data/raw/TCGA-PAAD_RNAseq_SE.rds



=> File saved



SummarizedExperiment: 60660 genes x 182 samples. Assays: unstranded, stranded_first, stranded_second, tpm_unstrand, fpkm_unstrand, fpkm_uq_unstrand
